<a href="https://colab.research.google.com/github/jamagiwa/L-Trail/blob/main/reproducibility/Fig3_bonemarrow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#L-Trail: Reproducing Figure3 (Bone Marrow dataset)


In [ ]:
!git clone https://github.com/jamagiwa/L-Trail.git

%cd L-Trail

!pip install -r requirements.txt

In [ ]:
import scvelo as scv
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.stats import gaussian_kde, skew
import random
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors
import seaborn as sns
from scipy.spatial.distance import cosine
from ltrail import pl
from ltrail import tl

In [ ]:
SEED = 34

random.seed(SEED)
np.random.seed(SEED)
sc.settings.seed = SEED
scv.settings.seed = SEED

In [ ]:
#Font settings for publication
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans', 'Bitstream Vera Sans']

# Figure size and font size settings
plt.rcParams['font.size'] = 8
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['figure.titlesize'] = 12

# Scanpy plotting configurations
sc.set_figure_params(
    scanpy=True,
    dpi=100,
    dpi_save=300,
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(6, 6),
    color_map='viridis'
    )

#1. Data Acquisition

Load the bone marrow dataset using the scVelo built-in library

In [ ]:
adata_bm = scv.datasets.bonemarrow()
adata_bm_velo = adata_bm.copy()

#2. scVelo Benchmark (Dynamical Mode)

In [ ]:
adata_bm_velo = adata_bm.copy()
scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.set_figure_params('scvelo')

In [ ]:
scv.pp.filter_genes(adata_bm_velo, min_shared_counts=30)
scv.pp.normalize_per_cell(adata_bm_velo)
sc.pp.log1p(adata_bm_velo)
sc.pp.highly_variable_genes(adata_bm_velo, n_top_genes=2000)

sc.tl.pca(adata_bm_velo)
sc.pp.neighbors(adata_bm_velo, n_pcs=30, n_neighbors=30)
sc.tl.umap(adata_bm_velo)

scv.pp.moments(adata_bm_velo, n_pcs=30, n_neighbors=30)

In [ ]:
#Explicitly compute PCA and UMAP embeddings
scv.tl.velocity(adata_bm_velo)
scv.tl.velocity_graph(adata_bm_velo)

In [ ]:
scv.pl.velocity_embedding_stream(adata_bm_velo,
                                 basis='X_umap',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='right margin',
                                 title='Bone marrow velocity(stochastic, umap)',
                                 show=False,
                                 )


scv.pl.velocity_embedding_stream(adata_bm_velo,
                                 basis='X_pca',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='right margin',
                                 title='Bone marrow velocity(stochasstic, pca)',
                                 show=False
                                 )

In [ ]:
scv.tl.recover_dynamics(adata_bm_velo)

In [ ]:
scv.tl.differential_kinetic_test(adata_bm_velo, groupby='clusters')

In [ ]:
bm_kinetic = adata_bm_velo.var.sort_values('fit_pval_kinetics').index[:10]
scv.get_df(adata_bm_velo[:, bm_kinetic], ['fit_diff_kinetics', 'fit_pval_kinetics'], precision=2)

In [ ]:
target_genes_bm = ['ACTN1', 'PLXDC2', 'MED12L', 'RAB27B', 'MALAT1', 'CCDC175', 'HSP90B1', 'YPEL5', 'MMRN1', 'PDE4B']
bm_kinetic = ['ACTN1', 'PLXDC2', 'MED12L', 'RAB27B', 'MALAT1', 'CCDC175', 'HSP90B1', 'YPEL5', 'MMRN1', 'PDE4B']

sc.pl.pca(adata_bm_velo, color=target_genes_bm, cmap='viridis', s=20)
sc.pl.umap(adata_bm_velo, color=target_genes_bm, cmap='viridis', s=20)

In [ ]:
kwargs = dict(linewidth=2, add_linfit=True, frameon=False)

scv.pl.scatter(adata_bm_velo, basis=bm_kinetic, add_outline='fit_diff_kinetics', **kwargs)

In [ ]:
diff_clusters_bm = list(adata_bm_velo[:, bm_kinetic].var['fit_diff_kinetics'])
scv.pl.scatter(adata_bm_velo,
               legend_loc='right',
               size=60,
               title='diff_kinetics_bm',
               add_outline=diff_clusters_bm,
               outline_width=(.8, .2))

In [ ]:
#Kineticsを考慮して再計算
scv.tl.velocity(adata_bm_velo, mode='dynamical', diff_kinetics=True)

scv.tl.velocity_graph(adata_bm_velo)

In [ ]:
bm_cellcounts = adata_bm_velo.obs['clusters'].value_counts()
labels = [f"{cluster}\n(n={count})" for cluster, count in zip(bm_cellcounts.index, bm_cellcounts.values)]

fig, ax = plt.subplots(figsize=(7, 5))

sns.barplot(
    x=bm_cellcounts.index,
    y=bm_cellcounts.values,
    order=bm_cellcounts.index,
    color='#4C72B0',
    edgecolor='black',
    linewidth=1,
    ax=ax
)

ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=11)
ax.tick_params(axis='y', labelsize=11)

ax.set_xlabel('Bone Marrow Clusters', fontsize=14, labelpad=10)
ax.set_ylabel('Number of Cells', fontsize=14, labelpad=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)

ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle='--', color='lightgrey', alpha=0.7)


plt.tight_layout()

In [ ]:
scv.pl.velocity_embedding_stream(adata_bm_velo,
                                 basis='umap',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='on data',
                                 title='Bone marrow RNA Velocity (Dynamical, umap)',
                                 show=False,
                                 )

scv.pl.velocity_embedding_stream(adata_bm_velo,
                                 basis='pca',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='on data',
                                 title='Bone marrow RNA Velocity (Dynamical, pca)',
                                 show=False
                                 )

#3. L-Trail Implementation

Compute and visualize macroscopic transition vectors using L-Trail.

In [ ]:
#pca
pl.plot_ltrail(adata_bm_velo,
            groupby='clusters',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Bone marrow L-moment Cluster(scale=30, p<0.05, pca)',
            show=False,
            figsize=(10, 8)
            )
#umap
pl.plot_ltrail(adata_bm_velo,
            groupby='clusters',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Bone marrow L-moment Cluster(scale=30, p<0.05, umap)',
            show=False,
            figsize=(10, 8)
            )

#4. Quantitative Evaluation of Trajectory Alignment

Local Trajectory Alignment via k-NN Sampling




In [ ]:
tl.calc_velocity_ltrail_similarity(adata_bm_velo,
                                groupby='clusters',
                                method='lmoment',
                                use_rep='X_pca',
                                vel_rep='velocity_pca',
                                )

Compute local cosine similarities between L-Trail vectors and RNA velocity vectors using randomly sampled anchor cells and their local neighborhoods.


In [ ]:
result_knn_cos_lmoment = tl.calc_knn_similarity(adata_bm_velo,
                                             use_rep='X_pca',
                                             vel_rep='velocity_pca',
                                             n_pcs=30,
                                             method='lmoment',
                                             k=50,
                                             n_anchors=1000,
                                             )

pl.plot_knn_similarity(adata_bm_velo,
                    result_knn_cos_lmoment,
                    basis='X_pca',
                    title='Bone marrow cos similarity (L-Moment vs Velocity)',
                    figsize=(7, 5),
                    show=False
                   )

pl.boxplot_similarity(adata_bm_velo,
                        result_knn_cos_lmoment,
                        groupby='clusters',
                        figsize=(7, 5),
                        ylim=(-1, 1),
                        title='Bone marrow cos similarity (L-Moment vs Velocity)',
                        show=False
                        )